In [6]:
import copy
import os
import math

import numpy as np
import torch
from torch import nn, Tensor
import torch.nn.functional as F
from matplotlib import pyplot as plt
import seaborn as sns
from typing import Literal

In [ ]:
def attention(Q, K, V):
    """Scaled dot product attention (notebook 3); broadcasts over any
    leading batch axes."""
    d = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d)
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights


class MultiHeadAttention(nn.Module):
    """Multi-head attention with learned projections (notebook 4)."""
    def __init__(self, num_hiddens: int, num_heads: int):
        super().__init__()

        self.num_heads = num_heads
        self.W_q = nn.Linear(num_hiddens, num_hiddens, bias=True)
        self.W_k = nn.Linear(num_hiddens, num_hiddens, bias=True)
        self.W_v = nn.Linear(num_hiddens, num_hiddens, bias=True)
        self.W_o = nn.Linear(num_hiddens, num_hiddens, bias=True)

    def split_heads(self, X: Tensor):
        """(batch, seq, num_hiddens) -> (batch, heads, seq, num_hiddens/heads)"""
        b, seq, _ = X.shape
        return X.reshape(b, seq, self.num_heads, -1).transpose(1, 2)

    def merge_heads(self, X: Tensor):
        """(batch, heads, seq, d_head) -> (batch, seq, num_hiddens)"""
        b, h, seq, d_head = X.shape
        return X.transpose(1, 2).reshape(b, seq, h * d_head)

    def forward(self, queries: Tensor, keys: Tensor, values: Tensor):
        """
        Input: (batch, seq, num_hiddens)
        Output: (batch, seq, num_hiddens)
        """
        # project, then split into heads: each is (batch, heads, seq, d_head)
        Q = self.split_heads(self.W_q(queries))   # (batch, heads, seq, d_head)
        K = self.split_heads(self.W_k(keys))
        V = self.split_heads(self.W_v(values))
        output, self.attention_weights = attention(Q, K, V)
        return self.W_o(self.merge_heads(output))


class PositionWiseFFN(nn.Module):
    """
    The same two-layer MLP applied at every position from class,
    but this time with GELU activation and dropout applied in the FFN.
    """
    def __init__(self, num_hiddens: int, ffn_num_hiddens: int, dropout: float):
        super().__init__()

        # nn.Linear acts on the last dimension, i.e., on each position separately.
        self.net = nn.Sequential(
            nn.Linear(num_hiddens, ffn_num_hiddens),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_num_hiddens, num_hiddens))


    def forward(self, X: Tensor):
        """
        Input: (batch, seq, num_hiddens)
        Output: (batch, seq, num_hiddens)
        """
        return self.net(X)


class AddNorm(nn.Module):
    def __init__(self, num_hiddens, dropout):
        super().__init__()

        # NOTE: the use of BatchNorm here instead of LayerNorm.
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.BatchNorm1d(num_hiddens)

    def forward(self, X: Tensor, Y: Tensor):
        """
        Input: X (batch, seq, num_hiddens),
               Y (batch, seq, num_hiddens)
        Output: (batch, seq, num_hiddens)
        """
        # X: the sublayer's input; Y: the sublayer's output
        return self.norm(self.dropout(Y) + X)


class TransformerEncoderBlock(nn.Module):
    def __init__(self, num_hiddens: int, ffn_num_hiddens: int, num_heads: int, dropout: float):
        super().__init__()
        self.attention = MultiHeadAttention(num_hiddens, num_heads)
        self.addnorm1 = AddNorm(num_hiddens, dropout)
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout)
        self.addnorm2 = AddNorm(num_hiddens, dropout)

    def forward(self, X: Tensor):
        """
        Input/Output: (batch * num_channels, num_patches, num_hiddens)
        """
        Y = self.addnorm1(X, self.attention(X, X, X))
        return self.addnorm2(Y, self.ffn(Y))


class PatchPositionalEmbedding(nn.Module):
    """
    Produce 'embeddings' of patches through:
    - A linear project that maps patches in dimension `p` to dimension `d = num_hiddens`.
    - A trained lookup matrix that encodes position of `N` patches.

    Output of this layer is flattened in the (batch x channel) dimension so input into
    MultiHeadAttention is the standard (samples, sequence, embed_d) shape.
    """
    def __init__(self, num_patches: int, patch_size: int, num_hiddens: int):
        super().__init__()
        self.W_patch = nn.Linear(patch_size, num_hiddens, bias=True)

        W_position = torch.empty((num_patches, num_hiddens))
        nn.init.uniform_(W_position, -0.02, 0.02)
        self.W_position = nn.Parameter(W_position)

    def forward(self, X: Tensor):
        """
        Input: (batch * num_channels, num_patches, patch_size)
        Output: (batch * num_channels, num_patches, num_hiddens)
        """
        return self.W_patch(X) + self.W_position


class Patcher(nn.Module):
    def __init__(self, patch_size: int, stride: int):
        self.patch_size = patch_size
        self.stride = stride
        self.padding = nn.ReplicationPad1d((0, stride))

    def forward(self, X: Tensor):
        """
        Input: (batch, lookback_size, num_channels)
        Output: (batch, num_channels, num_patches, patch_size)
        """
        X = X.permute(0, 2, 1)  # (batch, num_channels, lookback_size)
        return self.padding(X).unfold(dimension=-1, size=self.patch_size, step=self.stride)


class RevIN(nn.Module):
    """
    Reversible Instance Normalization: Normalize each time series (channel) with zero
    mean and a std dev of 1. Reverse this procedure when `'denorm'`ing.

    The official PatchTST implementation released with the paper uses the implementation
    from https://github.com/ts-kim/RevIN/blob/master/RevIN.py, copied here.

    MIT License

    Copyright (c) 2022 Electronics and Telecommunications Research Institute (ETRI)

    Permission is hereby granted, free of charge, to any person obtaining a copy
    of this software and associated documentation files (the "Software"), to deal
    in the Software without restriction, including without limitation the rights
    to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
    copies of the Software, and to permit persons to whom the Software is
    furnished to do so, subject to the following conditions:

    The above copyright notice and this permission notice shall be included in all
    copies or substantial portions of the Software.

    THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
    IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
    FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
    AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
    LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
    OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
    SOFTWARE.
    """
    def __init__(self, num_features: int, eps=1e-5, affine=True):
        """
        :param num_features: the number of features or channels
        :param eps: a value added for numerical stability
        :param affine: if True, RevIN has learnable affine parameters
        """
        super(RevIN, self).__init__()
        self.num_features = num_features
        self.eps = eps
        self.affine = affine
        if self.affine:
            self._init_params()

    def forward(self, x, mode: Literal["norm"]|Literal["denorm"]):
        if mode == 'norm':
            self._get_statistics(x)
            x = self._normalize(x)
        elif mode == 'denorm':
            x = self._denormalize(x)
        else: raise NotImplementedError
        return x

    def _init_params(self):
        # initialize RevIN params: (C,)
        self.affine_weight = nn.Parameter(torch.ones(self.num_features))
        self.affine_bias = nn.Parameter(torch.zeros(self.num_features))

    def _get_statistics(self, x):
        dim2reduce = tuple(range(1, x.ndim-1))
        self.mean = torch.mean(x, dim=dim2reduce, keepdim=True).detach()
        self.stdev = torch.sqrt(torch.var(x, dim=dim2reduce, keepdim=True, unbiased=False) + self.eps).detach()

    def _normalize(self, x):
        x = x - self.mean
        x = x / self.stdev
        if self.affine:
            x = x * self.affine_weight
            x = x + self.affine_bias
        return x

    def _denormalize(self, x):
        if self.affine:
            x = x - self.affine_bias
            x = x / (self.affine_weight + self.eps*self.eps)
        x = x * self.stdev
        x = x + self.mean
        return x


class PatchTST(nn.Module):
    """
    PatchTST architecture. Default parameter values are from the original paper.

    The default lookback window size of 512 time steps produces 64 patches with the
    default patch size of 16 and stride of 8. Another common variant is a
    lookback size of 336, producing 42 patches.

    The original paper tests models over prediction time horizons of 96, 192, 336,
    and 720. In the official source code, dropout is applied after positional encoding
    and in the transformer encoder block.
    """
    def __init__(self,
                num_channels: int,  # Number of individual time series in multivariate series.
                patch_size=16,
                stride=8,
                lookback_size=512,
                prediction_size=96,
                num_hiddens=128,
                ffn_num_hiddens=256,
                num_heads=16,
                num_blocks=3,
                dropout=0.2):
        super().__init__()
        self.num_patches = int((lookback_size - patch_size) / stride) + 2

        # Normalization layer standardizes each channel individually.
        self.norm_and_patch = nn.Sequential(
            RevIN(num_features=num_channels),
            Patcher(patch_size, stride))
        self.embedding = PatchPositionalEmbedding(self.num_patches, patch_size, num_hiddens)
        self.dropout = nn.Dropout(dropout)

        self.blks = nn.ModuleList([
            TransformerEncoderBlock(num_hiddens, ffn_num_hiddens, num_heads, dropout)
            for _ in range(num_blocks)
        ])

        self.decoder = nn.Sequential(
            nn.Flatten(start_dim=-2),
            nn.Linear(self.num_patches * num_hiddens, prediction_size))

    def forward(self, X: Tensor):
        """
        Input: (batch, num_channels, lookback_size)
        Output: (batch, num_channels, prediction_size)

        Each channel is one time series from the multivariate series.
        """
        batch, num_channels, _ = X.shape

        # (batch, num_channels, num_patches, patch_size)
        patched_X = self.norm_and_patch(X.transpose(1, 2))
        assert self.num_patches == patched_X.shape[2]

        # (batch * num_channels, num_patches, num_hiddens)
        # Batch and channels flattened together.
        X_batch_channel_flattened = X.flatten(start_dim=0, end_dim=1)
        patch_embeds: Tensor = self.dropout(self.embedding(X_batch_channel_flattened))

        for blk in self.blks:
            patch_embeds = blk(patch_embeds)

        # (batch, num_channels, num_patches, num_hiddens)
        # NOTE: The official code transposes the last two dimensions before feeding into
        # the decoder head. It shouldn't matter, but perhaps worth investigating if this
        # doesn't perform...
        patch_embeds = patch_embeds.unflatten(dim=0, sizes=(batch, num_channels))
        return self.decoder(patch_embeds)